# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
con.sql(f"""
    WITH grain_check AS (
        SELECT content_hash_id, report_date, COUNT(*) AS n
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
        GROUP BY content_hash_id, report_date
    )
    SELECT
        COUNT(*) AS unique_grain_combos,
        SUM(CASE WHEN n > 1 THEN 1 ELSE 0 END) AS duplicate_grain_groups,
        MAX(n) AS max_rows_in_one_group
    FROM grain_check
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────┬────────────────────────┬───────────────────────┐
│ unique_grain_combos │ duplicate_grain_groups │ max_rows_in_one_group │
│        int64        │         int128         │         int64         │
├─────────────────────┼────────────────────────┼───────────────────────┤
│             9841378 │                      0 │                     1 │
└─────────────────────┴────────────────────────┴───────────────────────┘

In [3]:
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT report_date) AS distinct_dates_present
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬────────────┬────────────────────────┐
│ total_rows │  min_date  │  max_date  │ distinct_dates_present │
│   int64    │    date    │    date    │         int64          │
├────────────┼────────────┼────────────┼────────────────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │                     31 │
└────────────┴────────────┴────────────┴────────────────────────┘

In [11]:
schema_df = con.sql(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    LIMIT 1
""").df()

for col in schema_df['column_name']:
    print(col)

report_date
client_hash_id
content_hash_id
client_has_gsc
client_has_ga4
gsc_data_available
ga4_data_available
gsc_impressions
gsc_clicks
gsc_sum_position
gsc_avg_position
ga4_pageviews
ga4_sessions
ga4_users
ga4_engaged_sessions
ga4_total_engagement_sec
sessions_organic
sessions_direct
sessions_referral
sessions_social
sessions_paid
sessions_ai
ai_chatgpt
ai_perplexity
ai_gemini
ai_copilot
ai_claude
ai_meta
ai_other
scroll_events
month


In [5]:
con.sql(f"""
    SELECT
        client_hash_id,
        COUNT(DISTINCT gsc_data_available) AS distinct_values
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY client_hash_id
    HAVING COUNT(DISTINCT gsc_data_available) > 1
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬─────────────────┐
│     client_hash_id      │ distinct_values │
│         varchar         │      int64      │
├─────────────────────────┼─────────────────┤
│ client_62f4a7e64f5e0096 │               2 │
│ client_9958f0a7ae1df715 │               2 │
│ client_73cda7b4e4f265ea │               2 │
│ client_fef1a8f436438636 │               2 │
│ client_08a6a72ff48e62c0 │               2 │
│ client_ba65e80a1116ae41 │               2 │
│ client_3ffa76342f366962 │               2 │
│ client_f623b01661d4bfe4 │               2 │
│ client_cd12bcfd98942aa1 │               2 │
│ client_a80fca3f171ed1de │               2 │
│            ·            │               · │
│            ·            │               · │
│            ·            │               · │
│ client_0fa64a184f18a4a0 │               2 │
│ client_86ebc2f12c01f586 │               2 │
│ client_0797ff3a1fc9a6a5 │               2 │
│ client_59256b0571e0c970 │               2 │
│ client_7eafe750768f0ac2 │       

In [6]:
con.sql(f"""
    SELECT
        gsc_data_available,
        COUNT(*) AS row_count,
        COUNT(gsc_impressions) AS non_null_impressions
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY gsc_data_available
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬───────────┬──────────────────────┐
│ gsc_data_available │ row_count │ non_null_impressions │
│      boolean       │   int64   │        int64         │
├────────────────────┼───────────┼──────────────────────┤
│ false              │   6230317 │              6230317 │
│ true               │   3611061 │              3611061 │
└────────────────────┴───────────┴──────────────────────┘

In [7]:
con.sql(f"""
    SELECT
        gsc_data_available,
        COUNT(*) AS row_count,
        SUM(CASE WHEN gsc_impressions = 0 THEN 1 ELSE 0 END) AS zero_impressions,
        ROUND(AVG(gsc_impressions), 2) AS avg_impressions
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY gsc_data_available
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬───────────┬──────────────────┬─────────────────┐
│ gsc_data_available │ row_count │ zero_impressions │ avg_impressions │
│      boolean       │   int64   │      int128      │     double      │
├────────────────────┼───────────┼──────────────────┼─────────────────┤
│ false              │   6230317 │          6230317 │             0.0 │
│ true               │   3611061 │                0 │           77.72 │
└────────────────────┴───────────┴──────────────────┴─────────────────┘

In [8]:
con.sql(f"""
    SELECT
        client_has_gsc,
        gsc_data_available,
        COUNT(*) AS row_count
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY client_has_gsc, gsc_data_available
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬────────────────────┬───────────┐
│ client_has_gsc │ gsc_data_available │ row_count │
│    boolean     │      boolean       │   int64   │
├────────────────┼────────────────────┼───────────┤
│ true           │ false              │   6230317 │
│ true           │ true               │   3611061 │
└────────────────┴────────────────────┴───────────┘

In [9]:
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS rows_available,
        ROUND(
            100.0 * SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS pct_available
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────┬───────────────┐
│ total_rows │ rows_available │ pct_available │
│   int64    │     int128     │    double     │
├────────────┼────────────────┼───────────────┤
│    3611061 │        3611061 │         100.0 │
└────────────┴────────────────┴───────────────┘

In [10]:
con.sql(f"""
    SELECT COUNT(*) AS rows_available
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ rows_available │
│     int64      │
├────────────────┤
│        3611061 │
└────────────────┘

In [12]:
con.sql(f"""
    SELECT
        ga4_data_available,
        COUNT(*) AS row_count,
        SUM(CASE WHEN scroll_events = 0 THEN 1 ELSE 0 END) AS zero_scroll_events,
        ROUND(AVG(scroll_events), 2) AS avg_scroll_events
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY ga4_data_available
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬───────────┬────────────────────┬───────────────────┐
│ ga4_data_available │ row_count │ zero_scroll_events │ avg_scroll_events │
│      boolean       │   int64   │       int128       │      double       │
├────────────────────┼───────────┼────────────────────┼───────────────────┤
│ NULL               │   3018741 │                  0 │              NULL │
│ false              │   6408671 │            6408671 │               0.0 │
│ true               │    413966 │             290926 │              0.53 │
└────────────────────┴───────────┴────────────────────┴───────────────────┘

In [13]:
con.sql(f"""
    SELECT
        ga4_data_available,
        COUNT(*) AS row_count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_march
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY ga4_data_available
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬───────────┬──────────────┐
│ ga4_data_available │ row_count │ pct_of_march │
│      boolean       │   int64   │    double    │
├────────────────────┼───────────┼──────────────┤
│ false              │   6408671 │        65.12 │
│ NULL               │   3018741 │        30.67 │
│ true               │    413966 │         4.21 │
└────────────────────┴───────────┴──────────────┘

In [14]:
con.sql(f"""
    SELECT
        ga4_data_available,
        COUNT(*) AS row_count,
        ROUND(AVG(ga4_pageviews), 2) AS avg_pageviews,
        ROUND(AVG(ga4_sessions), 2) AS avg_ga4_sessions,
        ROUND(AVG(ga4_users), 2) AS avg_users,
        ROUND(AVG(ga4_engaged_sessions), 2) AS avg_engaged_sessions,
        ROUND(AVG(ga4_total_engagement_sec), 2) AS avg_engagement_sec,
        ROUND(AVG(sessions_organic), 2) AS avg_sessions_organic,
        ROUND(AVG(sessions_paid), 2) AS avg_sessions_paid
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY ga4_data_available
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬───────────┬───────────────┬──────────────────┬───────────┬──────────────────────┬────────────────────┬──────────────────────┬───────────────────┐
│ ga4_data_available │ row_count │ avg_pageviews │ avg_ga4_sessions │ avg_users │ avg_engaged_sessions │ avg_engagement_sec │ avg_sessions_organic │ avg_sessions_paid │
│      boolean       │   int64   │    double     │      double      │  double   │        double        │       double       │        double        │      double       │
├────────────────────┼───────────┼───────────────┼──────────────────┼───────────┼──────────────────────┼────────────────────┼──────────────────────┼───────────────────┤
│ NULL               │   3018741 │          NULL │             NULL │      NULL │                 NULL │               NULL │                 NULL │              NULL │
│ false              │   6408671 │           0.0 │              0.0 │       0.0 │                  0.0 │                0.0 │                  0.0 │       

In [15]:
con.sql(f"""
    SELECT
        CORR(ga4_engaged_sessions, ga4_total_engagement_sec) AS corr
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND ga4_data_available IS TRUE
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┐
│        corr        │
│       double       │
├────────────────────┤
│ 0.6320979012436048 │
└────────────────────┘

In [16]:
con.sql(f"""
    SELECT
        report_date,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS available_rows,
        ROUND(100.0 * SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_available
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY report_date
    ORDER BY report_date
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────┬────────────────┬───────────────┐
│ report_date │ total_rows │ available_rows │ pct_available │
│    date     │   int64    │     int128     │    double     │
├─────────────┼────────────┼────────────────┼───────────────┤
│ 2026-03-01  │     275874 │         101910 │         36.94 │
│ 2026-03-02  │     276269 │         103696 │         37.53 │
│ 2026-03-03  │     311676 │         107362 │         34.45 │
│ 2026-03-04  │     311675 │         109377 │         35.09 │
│ 2026-03-05  │     311676 │         109740 │         35.21 │
│ 2026-03-06  │     312187 │         110037 │         35.25 │
│ 2026-03-07  │     312387 │         102153 │          32.7 │
│ 2026-03-08  │     313374 │         101170 │         32.28 │
│ 2026-03-09  │     313874 │         111313 │         35.46 │
│ 2026-03-10  │     314047 │         113051 │          36.0 │
│     ·       │        ·   │            ·   │            ·  │
│     ·       │        ·   │            ·   │            ·  │
│     · 

In [17]:
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_avg_position = 0 THEN 1 ELSE 0 END) AS zero_position_rows,
        MIN(gsc_avg_position) AS min_position,
        MAX(gsc_avg_position) AS max_position
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬──────────────┬──────────────┐
│ total_rows │ zero_position_rows │ min_position │ max_position │
│   int64    │       int128       │    double    │    double    │
├────────────┼────────────────────┼──────────────┼──────────────┤
│    3611061 │             163189 │          0.0 │        498.0 │
└────────────┴────────────────────┴──────────────┴──────────────┘

In [18]:
con.sql(f"""
    SELECT
        report_date,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS available_rows,
        ROUND(
            100.0 * SUM(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS pct_available
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY report_date
    ORDER BY report_date
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────┬────────────────┬───────────────┐
│ report_date │ total_rows │ available_rows │ pct_available │
│    date     │   int64    │     int128     │    double     │
├─────────────┼────────────┼────────────────┼───────────────┤
│ 2026-03-01  │     275874 │           4909 │          1.78 │
│ 2026-03-02  │     276269 │           5349 │          1.94 │
│ 2026-03-03  │     311676 │           6831 │          2.19 │
│ 2026-03-04  │     311675 │           6791 │          2.18 │
│ 2026-03-05  │     311676 │           7653 │          2.46 │
│ 2026-03-06  │     312187 │           8389 │          2.69 │
│ 2026-03-07  │     312387 │           7865 │          2.52 │
│ 2026-03-08  │     313374 │           7589 │          2.42 │
│ 2026-03-09  │     313874 │           9653 │          3.08 │
│ 2026-03-10  │     314047 │          11731 │          3.74 │
│     ·       │        ·   │            ·   │            ·  │
│     ·       │        ·   │            ·   │            ·  │
│     · 

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.